In [2]:
import sys
sys.path.append('/home/ec2-user/SEAJ-TEAM-4-')

In [3]:
import pandas as pd 
import matplotlib.pyplot as plt 

from backend.data_processing.constants import DEFAULT_TICKERS
from backend.data_analysis.loader import MarketDataLoader
from backend.data_analysis.cleaner import MarketDataCleaner

loader = MarketDataLoader()
raw_df = loader.load_analysis_data(DEFAULT_TICKERS)

print("Raw Shape:", raw_df.shape)

2026-09-15 07:34:41,973 - INFO - Starting data load for 47 tickers
2026-09-15 07:34:41,974 - INFO - Downloading batch 1 with 47 tickers
2026-09-15 07:34:41,975 - INFO - Downloading historical data for 47 instruments from 2016-01-01 to None
2026-09-15 07:34:46,289 - INFO - Successfully downloaded data for batch of 47 tickers
2026-09-15 07:34:46,577 - INFO - Converted to long format: 126383 records, 7 columns
2026-09-15 07:34:46,580 - INFO - Successfully downloaded data for 126383 records across all batches
2026-09-15 07:36:59,207 - ERROR - Failed to load instrument metadata: connection to server at "10.9.69.224", port 5432 failed: Connection timed out
	Is the server running on that host and accepting TCP/IP connections?

2026-09-15 07:36:59,209 - WARNING - No metadata retrieved from PostgreSQL, returning market data only


Raw Shape: (126383, 7)


## Quality Report

In [21]:
quality = MarketDataCleaner.generate_quality_report(raw_df)
print("Data Quality Report:")
quality


2026-09-15 07:49:59,548 - WARNING - Found 1 rows with invalid volumes (volume <= 0)
2026-09-15 07:49:59,553 - WARNING - Found 28 rows with invalid OHLC relationships
2026-09-15 07:49:59,572 - INFO - === Data Quality Report ===
2026-09-15 07:49:59,573 - INFO - Total records: 126383
2026-09-15 07:49:59,574 - INFO - Unique symbols: 47
2026-09-15 07:49:59,576 - INFO - Issues found: 29


Data Quality Report:


{'total_records': 126383,
 'total_symbols': 47,
 'date_range': {'min': '2016-01-04 00:00:00', 'max': '2026-09-14 00:00:00'},
 'duplicates': 0,
 'missing_symbols': np.int64(0),
 'missing_dates': np.int64(0),
 'invalid_prices': 0,
 'invalid_volumes': 1,
 'invalid_ohlc': 28,
 'missing_values_summary': {'missing_count': {}, 'missing_percentage': {}}}

## Missing Values

In [5]:
missing = MarketDataCleaner.detect_missing_values(raw_df)
missing

,missing_count,missing_percentage


In [6]:
missing_nonzero = missing[missing['missing_count'] > 0]
missing_nonzero

,missing_count,missing_percentage


In [7]:
if not missing_nonzero.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    missing_nonzero.plot(kind='bar', ax=ax)
    ax.set_title("Missing Nonzero Values")
    ax.set_xlabel("Columns")
    ax.set_ylabel("Count")
    plt.show()
else:
    print("No missing nonzero values found.")
    


No missing nonzero values found.


## Duplicates

In [12]:
duplicates = MarketDataCleaner.detect_duplicates(raw_df)

print("Duplicate Rows:", len(duplicates))
duplicates.head(20)


Duplicate Rows: 0


,symbol,date,open,high,low,close,volume


## Invalid Prices

In [ ]:
invalid_prices = MarketDataCleaner.detect_invalid_prices(raw_df)

print("Invalid Prices:", len(invalid_prices))
invalid_prices.head(20)

Invalid Prices: 0


,symbol,date,open,high,low,close,volume


## Invalid Volume

In [ ]:
invalid_volumes = MarketDataCleaner.detect_invalid_volumes(raw_df)

print("Rows with Invalid Volumes:", len(invalid_volumes))
invalid_volumes.head(20)

2026-09-15 07:41:03,520 - WARNING - Found 1 rows with invalid volumes (volume <= 0)


Rows with Invalid Volumes: 1


,symbol,date,open,high,low,close,volume
108205,BND,2018-07-26,61.846809,61.846809,61.846809,61.846809,0


## OHLC Consistency

In [ ]:
invalid_ohlc = MarketDataCleaner.detect_invalid_ohlc(raw_df)

print("Rows with inconsistent OHLC:", len(invalid_ohlc))
invalid_ohlc.head(20)

2026-09-15 07:42:05,331 - WARNING - Found 28 rows with invalid OHLC relationships


Rows with inconsistent OHLC: 28


,symbol,date,open,high,low,close,volume
22507,BAC,2019-12-16,29.688861,29.807785,29.476494,29.476494,50777100
25253,WFC,2020-03-10,29.501495,29.884853,27.499515,29.884853,40078700
26904,GS,2016-01-25,125.621467,125.725888,121.380539,121.380539,5051700
38397,V,2018-12-27,121.167117,124.972816,119.775479,124.972816,10883000
40851,JNJ,2018-01-22,115.714115,116.603561,115.485856,116.603561,7010400
45745,KO,2016-02-19,31.406353,31.543278,31.190153,31.543278,12850000
45753,KO,2016-03-02,31.327080,31.543278,31.291045,31.543278,12856300
45937,KO,2016-11-21,30.184559,30.516582,30.051750,30.516582,12307000
56541,HD,2016-04-18,104.553836,106.097214,104.499553,106.097214,3964300
61907,XOM,2016-03-31,53.666627,54.324134,53.360214,53.360214,13896900


## Invalid Dates

In [ ]:
missing_dates_count = MarketDataCleaner.detect_missing_dates(raw_df)

print("Rows with Missing Dates:", missing_dates_count)
rows_with_missing_dates = raw_df[raw_df["date"].isna()]
rows_with_missing_dates.head(20)

Rows with Missing Dates: 0


,symbol,date,open,high,low,close,volume


## Apply Cleaning

In [ ]:
clean_df = MarketDataCleaner.clean_data(raw_df)

print("Before cleaning:", raw_df.shape)
print("After cleaning:", clean_df.shape)
print("Rows removed:", len(raw_df) - len(clean_df))
